# 🧠 ASTE with DeBERTa-v3-base
### Aspect-Sentiment Triplet Extraction (Valence-Arousal)
**Architecture**: DeBERTa-v3-base → BIO Extraction Head + VA Regression Head

---
**Data Paths** (edit these if needed):
- `LAPTOP_TRAIN` → laptop training JSONL
- `RESTAURANT_TRAIN` → restaurant training JSONL
- `TEST_FILE` → test JSONL (released Monday)
- `OUTPUT_DIR` → Google Drive output folder

In [ ]:
# ─────────────────────────────────────────────────────────────
#  SECTION 0 — CONFIGURATION  (edit only this cell)
# ─────────────────────────────────────────────────────────────

# ── Data paths ───────────────────────────────────────────────
LAPTOP_TRAIN     = "/content/drive/MyDrive/nlp_assignment/laptop_train.jsonl"
RESTAURANT_TRAIN = "/content/drive/MyDrive/nlp_assignment/restaurant_train.jsonl"
TEST_FILE        = "/content/drive/MyDrive/nlp_assignment/test.jsonl"   # released Monday

# ── Output ───────────────────────────────────────────────────
OUTPUT_DIR       = "/content/drive/MyDrive/nlp_assignment/outputs"
PRED_FILE        = f"{OUTPUT_DIR}/predictions.jsonl"
CHECKPOINT_DIR   = f"{OUTPUT_DIR}/checkpoints"

# ── Model ────────────────────────────────────────────────────
MODEL_NAME       = "microsoft/deberta-v3-base"

# ── Hyperparameters (tuned for high F1) ──────────────────────
MAX_LEN          = 128
BATCH_SIZE       = 16          # reduce to 8 if OOM
GRAD_ACCUM       = 2           # effective batch = 32
EPOCHS           = 15
LR               = 2e-5
WEIGHT_DECAY     = 0.01
WARMUP_RATIO     = 0.1
LAMBDA_CE        = 1.0         # BIO loss weight
LAMBDA_MSE       = 0.5         # VA regression loss weight
DROPOUT          = 0.1
SEED             = 42
VAL_SPLIT        = 0.1         # 10 % held out for validation
PATIENCE         = 5           # early stopping patience (epochs)
FP16             = True        # mixed precision (requires GPU)

# ── BIO label map ────────────────────────────────────────────
LABEL2ID = {"O": 0, "B-ASP": 1, "I-ASP": 2, "B-OPT": 3, "I-OPT": 4}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)
print("Config loaded ✓")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  SECTION 1 — MOUNT DRIVE & INSTALL DEPS
# ─────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Drive mounted ✓")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  SECTION 2 — INSTALL PACKAGES
# ─────────────────────────────────────────────────────────────
!pip install -q transformers==4.40.0 accelerate sentencepiece protobuf
print("Packages installed ✓")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  SECTION 3 — IMPORTS
# ─────────────────────────────────────────────────────────────
import json, random, warnings, copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
)
from collections import defaultdict
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  SECTION 4 — DATA LOADING & PREPROCESSING
# ─────────────────────────────────────────────────────────────

def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def get_char_spans(text, term):
    """Return all (start, end) char offsets of `term` in `text`."""
    spans = []
    if not term or term.strip().upper() == "NULL":
        return spans
    start = 0
    while True:
        idx = text.find(term, start)
        if idx == -1:
            break
        spans.append((idx, idx + len(term)))
        start = idx + 1
    return spans

def parse_va(va_str):
    """Parse '6.75#6.38' → (6.75, 6.38). Clamp to [1,9]."""
    try:
        v, a = va_str.split("#")
        return max(1.0, min(9.0, float(v))), max(1.0, min(9.0, float(a)))
    except:
        return 5.0, 5.0

def normalize_va(val, lo=1.0, hi=9.0):
    """Normalize VA score to [0,1] for sigmoid output."""
    return (val - lo) / (hi - lo)

def denormalize_va(val, lo=1.0, hi=9.0):
    """Convert sigmoid output back to [1,9]."""
    return round(val * (hi - lo) + lo, 2)

# ── Load and merge datasets ───────────────────────────────────
laptop_data = load_jsonl(LAPTOP_TRAIN)
restaurant_data = load_jsonl(RESTAURANT_TRAIN)
all_data = laptop_data + restaurant_data
random.shuffle(all_data)

print(f"Laptop train samples    : {len(laptop_data)}")
print(f"Restaurant train samples: {len(restaurant_data)}")
print(f"Total merged samples    : {len(all_data)}")

# ── Train / Val split ────────────────────────────────────────
n_val = int(len(all_data) * VAL_SPLIT)
val_data   = all_data[:n_val]
train_data = all_data[n_val:]
print(f"Train: {len(train_data)}  |  Val: {len(val_data)}")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  SECTION 5 — TOKENIZER
# ─────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME} ✓")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  SECTION 6 — DATASET CLASS (BIO Label Alignment)
# ─────────────────────────────────────────────────────────────

class ASTEDataset(Dataset):
    def __init__(self, records, tokenizer, max_len, is_test=False):
        self.records   = records
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.is_test   = is_test

    def __len__(self):
        return len(self.records)

    def _align_labels(self, text, quadruplets, offset_mapping, seq_len):
        """
        Align char-level spans to token indices using offset_mapping.
        Returns:
          bio_labels : (seq_len,) int tensor with BIO tags
          va_target  : (2,) float tensor — mean VA across all non-NULL quadruplets
        """
        bio_labels = [LABEL2ID["O"]] * seq_len

        va_values = []
        for q in quadruplets:
            asp  = q.get("Aspect", "NULL")
            opt  = q.get("Opinion", "NULL")
            va   = q.get("VA", "5.0#5.0")

            v_norm, a_norm = [normalize_va(x) for x in parse_va(va)]
            va_values.append((v_norm, a_norm))

            # ── Aspect spans ──────────────────────────────────
            for char_start, char_end in get_char_spans(text, asp):
                first = True
                for tok_idx, (os, oe) in enumerate(offset_mapping):
                    if os == 0 and oe == 0:
                        continue  # special tokens
                    if oe <= char_start or os >= char_end:
                        continue
                    if first:
                        bio_labels[tok_idx] = LABEL2ID["B-ASP"]
                        first = False
                    else:
                        bio_labels[tok_idx] = LABEL2ID["I-ASP"]

            # ── Opinion spans ─────────────────────────────────
            for char_start, char_end in get_char_spans(text, opt):
                first = True
                for tok_idx, (os, oe) in enumerate(offset_mapping):
                    if os == 0 and oe == 0:
                        continue
                    if oe <= char_start or os >= char_end:
                        continue
                    if first:
                        bio_labels[tok_idx] = LABEL2ID["B-OPT"]
                        first = False
                    else:
                        bio_labels[tok_idx] = LABEL2ID["I-OPT"]

        # ── VA target: mean across all quadruplets ────────────
        if va_values:
            v_target = np.mean([x[0] for x in va_values])
            a_target = np.mean([x[1] for x in va_values])
        else:
            v_target, a_target = 0.5, 0.5  # neutral

        return bio_labels, [v_target, a_target]

    def __getitem__(self, idx):
        rec  = self.records[idx]
        text = rec["Text"]
        sample_id = rec.get("ID", str(idx))

        enc = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_offsets_mapping=True,
            return_tensors="pt",
        )

        input_ids      = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        token_type_ids = enc.get("token_type_ids", torch.zeros_like(input_ids)).squeeze(0)
        offset_mapping = enc["offset_mapping"].squeeze(0).tolist()
        seq_len        = input_ids.size(0)

        if self.is_test:
            return {
                "input_ids":      input_ids,
                "attention_mask": attention_mask,
                "token_type_ids": token_type_ids,
                "offset_mapping": torch.tensor(offset_mapping),
                "id":   sample_id,
                "text": text,
            }

        quadruplets = rec.get("Quadruplet", [])
        bio_labels, va_target = self._align_labels(text, quadruplets, offset_mapping, seq_len)

        return {
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "token_type_ids": token_type_ids,
            "offset_mapping": torch.tensor(offset_mapping),
            "bio_labels":     torch.tensor(bio_labels, dtype=torch.long),
            "va_target":      torch.tensor(va_target,  dtype=torch.float),
            "id":   sample_id,
            "text": text,
            "quadruplets": quadruplets,
        }


def collate_fn(batch):
    keys = [k for k in batch[0] if k not in ("id", "text", "quadruplets", "offset_mapping")]
    out = {k: torch.stack([b[k] for b in batch]) for k in keys if isinstance(batch[0][k], torch.Tensor)}
    out["offset_mapping"] = torch.stack([b["offset_mapping"] for b in batch])
    out["ids"]   = [b["id"]   for b in batch]
    out["texts"] = [b["text"] for b in batch]
    if "quadruplets" in batch[0]:
        out["quadruplets"] = [b["quadruplets"] for b in batch]
    return out


train_ds = ASTEDataset(train_data, tokenizer, MAX_LEN)
val_ds   = ASTEDataset(val_data,   tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  SECTION 7 — MODEL ARCHITECTURE
#  DeBERTa-v3-base → BIO Head + VA Regression Head
# ─────────────────────────────────────────────────────────────

class ASTEModel(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size  # 768 for DeBERTa-v3-base

        # ── Extraction Head (Token Classification / BIO) ──────
        self.extraction_dropout = nn.Dropout(dropout)
        self.extraction_head    = nn.Linear(hidden, num_labels)

        # ── Regression Head (MLP on [CLS] → VA) ──────────────
        self.regression_head = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 2),
            nn.Sigmoid(),          # outputs in [0,1], scaled to [1,9] at decode
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids if token_type_ids is not None else None,
        )
        seq_out = outputs.last_hidden_state       # (B, seq_len, hidden)
        cls_out = seq_out[:, 0, :]               # (B, hidden)

        # BIO logits
        bio_logits = self.extraction_head(self.extraction_dropout(seq_out))  # (B, seq_len, num_labels)

        # VA scores (sigmoid → [0,1])
        va_scores  = self.regression_head(cls_out)   # (B, 2)

        return bio_logits, va_scores


model = ASTEModel(MODEL_NAME, NUM_LABELS, DROPOUT).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model loaded. Trainable parameters: {n_params:,}")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  SECTION 8 — OPTIMIZER, SCHEDULER, LOSS
# ─────────────────────────────────────────────────────────────

# Layer-wise LR decay (encoder gets lower LR than heads)
no_decay = ["bias", "LayerNorm.weight"]
optimizer_grouped_params = [
    {"params": [p for n, p in model.encoder.named_parameters()
                if not any(nd in n for nd in no_decay)],
     "lr": LR, "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in model.encoder.named_parameters()
                if any(nd in n for nd in no_decay)],
     "lr": LR, "weight_decay": 0.0},
    {"params": list(model.extraction_head.parameters()) +
               list(model.regression_head.parameters()) +
               list(model.extraction_dropout.parameters()),
     "lr": LR * 10, "weight_decay": WEIGHT_DECAY},
]

optimizer = torch.optim.AdamW(optimizer_grouped_params)

total_steps   = (len(train_loader) // GRAD_ACCUM) * EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)
scheduler     = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler        = GradScaler(enabled=FP16 and DEVICE.type == "cuda")

ce_loss_fn  = nn.CrossEntropyLoss(ignore_index=-100)
mse_loss_fn = nn.MSELoss()

print(f"Total steps: {total_steps} | Warmup: {warmup_steps}")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  SECTION 9 — SPAN DECODE & TRIPLET FORMATION
# ─────────────────────────────────────────────────────────────

def extract_spans_from_bio(bio_seq, id2label):
    """
    Given a list of predicted label ids, return
    {'ASP': [(start_tok, end_tok), ...], 'OPT': [...]}
    end_tok is INCLUSIVE.
    """
    spans = defaultdict(list)
    cur_type, cur_start = None, None

    for i, lid in enumerate(bio_seq):
        tag = id2label[lid]
        if tag.startswith("B-"):
            if cur_type is not None:
                spans[cur_type].append((cur_start, i - 1))
            cur_type  = tag[2:]
            cur_start = i
        elif tag.startswith("I-") and cur_type == tag[2:]:
            pass  # continue current span
        else:
            if cur_type is not None:
                spans[cur_type].append((cur_start, i - 1))
            cur_type, cur_start = None, None

    if cur_type is not None:
        spans[cur_type].append((cur_start, len(bio_seq) - 1))
    return spans


def tokens_to_text(token_ids, offsets, text):
    """Reconstruct span text from token offsets and original text."""
    valid_offsets = [(s, e) for (s, e) in offsets if not (s == 0 and e == 0)]
    if not valid_offsets:
        return ""
    char_start = valid_offsets[0][0]
    char_end   = valid_offsets[-1][1]
    return text[char_start:char_end]


def span_indices_to_text(start_tok, end_tok, offset_mapping, text):
    """Convert token span [start_tok, end_tok] → substring of text."""
    offsets = offset_mapping[start_tok: end_tok + 1]
    valid   = [(s, e) for (s, e) in offsets if not (s == 0 and e == 0)]
    if not valid:
        return None
    return text[valid[0][0]: valid[-1][1]].strip()


def form_triplets(bio_preds, va_pred, offset_mapping, text):
    """
    Build (Aspect, Opinion, VA) triplets from BIO predictions.
    Handles NULL cases:
      - no ASP spans → Aspect = NULL
      - no OPT spans → Opinion = NULL
    Pairing: zip ASP & OPT spans; if unequal, extend shorter with NULL.
    """
    v_score = denormalize_va(va_pred[0])
    a_score = denormalize_va(va_pred[1])
    va_str  = f"{v_score:.2f}#{a_score:.2f}"

    spans = extract_spans_from_bio(bio_preds, ID2LABEL)
    asp_spans = spans.get("ASP", [])
    opt_spans = spans.get("OPT", [])

    # Pair spans
    max_len = max(len(asp_spans), len(opt_spans), 1)
    triplets = []

    for i in range(max_len):
        asp_text = None
        if i < len(asp_spans):
            asp_text = span_indices_to_text(*asp_spans[i], offset_mapping, text)

        opt_text = None
        if i < len(opt_spans):
            opt_text = span_indices_to_text(*opt_spans[i], offset_mapping, text)

        triplets.append({
            "Aspect":  asp_text if asp_text else "NULL",
            "Opinion": opt_text if opt_text else "NULL",
            "VA":      va_str,
        })

    if not triplets:
        triplets = [{"Aspect": "NULL", "Opinion": "NULL", "VA": va_str}]

    return triplets


print("Decode utilities ready ✓")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  SECTION 10 — EVALUATION METRICS (F1 for ASTE)
# ─────────────────────────────────────────────────────────────

def triplet_f1(pred_triplets_batch, gold_triplets_batch):
    """
    Macro-averaged F1 over (Aspect, Opinion) pairs.
    VA is regression, not counted in F1.
    A predicted triplet is correct if (Aspect.lower, Opinion.lower) match gold.
    NULL is treated as a valid value.
    """
    n_pred = n_gold = n_match = 0

    for preds, golds in zip(pred_triplets_batch, gold_triplets_batch):
        gold_pairs = set(
            (g.get("Aspect",  "NULL").lower().strip(),
             g.get("Opinion", "NULL").lower().strip())
            for g in golds
        )
        pred_pairs = set(
            (p["Aspect"].lower().strip(),
             p["Opinion"].lower().strip())
            for p in preds
        )
        n_pred  += len(pred_pairs)
        n_gold  += len(gold_pairs)
        n_match += len(pred_pairs & gold_pairs)

    prec = n_match / n_pred  if n_pred  > 0 else 0.0
    rec  = n_match / n_gold  if n_gold  > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return {"precision": prec, "recall": rec, "f1": f1}

print("Metrics ready ✓")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  SECTION 11 — TRAINING LOOP
# ─────────────────────────────────────────────────────────────

def run_epoch(model, loader, optimizer, scheduler, scaler, training=True):
    model.train(training)
    total_loss = total_ce = total_mse = 0.0
    all_pred_triplets = []
    all_gold_triplets = []

    optimizer.zero_grad()
    pbar = tqdm(loader, desc="Train" if training else "Val", leave=False)

    for step, batch in enumerate(pbar):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        token_type_ids = batch.get("token_type_ids")
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(DEVICE)

        with autocast(enabled=FP16 and DEVICE.type == "cuda"):
            bio_logits, va_scores = model(input_ids, attention_mask, token_type_ids)

            if training:
                bio_labels = batch["bio_labels"].to(DEVICE)   # (B, seq)
                va_target  = batch["va_target"].to(DEVICE)    # (B, 2)

                # Mask padding tokens in CE loss
                active_mask  = attention_mask.view(-1).bool()
                active_logits = bio_logits.view(-1, NUM_LABELS)[active_mask]
                active_labels = bio_labels.view(-1)[active_mask]

                ce_loss  = ce_loss_fn(active_logits, active_labels)
                mse_loss = mse_loss_fn(va_scores, va_target)
                loss     = LAMBDA_CE * ce_loss + LAMBDA_MSE * mse_loss

                scaler.scale(loss / GRAD_ACCUM).backward()

                if (step + 1) % GRAD_ACCUM == 0:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad()

                total_loss += loss.item()
                total_ce   += ce_loss.item()
                total_mse  += mse_loss.item()

        # ── Decode for F1 ─────────────────────────────────────
        with torch.no_grad():
            bio_preds  = bio_logits.argmax(-1).cpu().tolist()   # (B, seq)
            va_preds   = va_scores.detach().cpu().tolist()       # (B, 2)
            offsets    = batch["offset_mapping"].tolist()
            texts      = batch["texts"]

            for b in range(len(texts)):
                pred_t = form_triplets(bio_preds[b], va_preds[b], offsets[b], texts[b])
                all_pred_triplets.append(pred_t)

        if not training and "quadruplets" in batch:
            all_gold_triplets.extend(batch["quadruplets"])

        pbar.set_postfix(loss=f"{total_loss/(step+1):.4f}")

    n = len(loader)
    metrics = {"loss": total_loss / n, "ce": total_ce / n, "mse": total_mse / n}
    if not training and all_gold_triplets:
        metrics.update(triplet_f1(all_pred_triplets, all_gold_triplets))
    return metrics


# ── Training with early stopping ─────────────────────────────
best_f1         = 0.0
best_state_dict = None
patience_ctr    = 0
history         = []

print("\n🚀 Starting training...")
for epoch in range(1, EPOCHS + 1):
    train_m = run_epoch(model, train_loader, optimizer, scheduler, scaler, training=True)
    val_m   = run_epoch(model, val_loader,   optimizer, scheduler, scaler, training=False)

    f1 = val_m.get("f1", 0.0)
    history.append({"epoch": epoch, **train_m, **{f"val_{k}": v for k, v in val_m.items()}})

    print(f"Epoch {epoch:02d}/{EPOCHS}  "
          f"train_loss={train_m['loss']:.4f}  "
          f"val_loss={val_m['loss']:.4f}  "
          f"val_F1={f1:.4f}  "
          f"val_P={val_m.get('precision',0):.4f}  "
          f"val_R={val_m.get('recall',0):.4f}")

    # Save best
    if f1 > best_f1:
        best_f1 = f1
        best_state_dict = copy.deepcopy(model.state_dict())
        ckpt_path = f"{CHECKPOINT_DIR}/best_model.pt"
        torch.save(best_state_dict, ckpt_path)
        print(f"  ✅ New best F1={best_f1:.4f} — checkpoint saved.")
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"  ⏹ Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs).")
            break

print(f"\n✅ Training complete. Best Val F1 = {best_f1:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  SECTION 12 — RELOAD BEST CHECKPOINT
# ─────────────────────────────────────────────────────────────
model.load_state_dict(torch.load(f"{CHECKPOINT_DIR}/best_model.pt", map_location=DEVICE))
model.eval()
print(f"Best model loaded (Val F1 = {best_f1:.4f}) ✓")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  SECTION 13 — GENERATE PREDICTIONS ON TEST SET
# ─────────────────────────────────────────────────────────────

test_data = load_jsonl(TEST_FILE)
print(f"Test samples: {len(test_data)}")

test_ds     = ASTEDataset(test_data, tokenizer, MAX_LEN, is_test=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=collate_fn, num_workers=2, pin_memory=True)

predictions = []   # list of dicts for output JSONL

model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        token_type_ids = batch.get("token_type_ids")
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(DEVICE)

        with autocast(enabled=FP16 and DEVICE.type == "cuda"):
            bio_logits, va_scores = model(input_ids, attention_mask, token_type_ids)

        bio_preds = bio_logits.argmax(-1).cpu().tolist()
        va_preds  = va_scores.cpu().tolist()
        offsets   = batch["offset_mapping"].tolist()
        texts     = batch["texts"]
        ids       = batch["ids"]

        for b in range(len(texts)):
            triplets = form_triplets(bio_preds[b], va_preds[b], offsets[b], texts[b])
            predictions.append({
                "ID":     ids[b],
                "Triplet": triplets,
            })

print(f"\nPredictions generated: {len(predictions)} samples")

# ── Write to Drive ────────────────────────────────────────────
with open(PRED_FILE, "w") as f:
    for rec in predictions:
        f.write(json.dumps(rec) + "\n")

print(f"\n💾 Saved predictions → {PRED_FILE}")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  SECTION 14 — VERIFY OUTPUT FORMAT
# ─────────────────────────────────────────────────────────────
print("=== First 5 predictions ===")
with open(PRED_FILE) as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        rec = json.loads(line)
        print(json.dumps(rec, indent=2))
        print()

In [ ]:
# ─────────────────────────────────────────────────────────────
#  SECTION 15 — TRAINING HISTORY SUMMARY
# ─────────────────────────────────────────────────────────────
print(f"{'Epoch':>6}  {'TrainLoss':>10}  {'ValLoss':>8}  {'ValF1':>7}  {'ValPrec':>8}  {'ValRec':>7}")
print("-" * 55)
for h in history:
    print(f"{h['epoch']:>6}  {h['loss']:>10.4f}  "
          f"{h.get('val_loss',0):>8.4f}  "
          f"{h.get('val_f1',0):>7.4f}  "
          f"{h.get('val_precision',0):>8.4f}  "
          f"{h.get('val_recall',0):>7.4f}")

print(f"\n🏆 Best Val F1: {best_f1:.4f}")
print(f"📁 Checkpoint : {CHECKPOINT_DIR}/best_model.pt")
print(f"📄 Predictions: {PRED_FILE}")